In [9]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 1 — Detect platform + set paths                   ║
# ╚══════════════════════════════════════════════════════════╝

import os, sys

ON_KAGGLE = os.path.exists('/kaggle/working')
ON_COLAB  = 'google.colab' in sys.modules
PLATFORM  = 'kaggle' if ON_KAGGLE else 'colab'

WORK_DIR  = '/kaggle/working' if ON_KAGGLE else '/content'
CKPT_DIR  = f'{WORK_DIR}/checkpoints'
AUDIO_DIR = f'{WORK_DIR}/audio'

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(AUDIO_DIR, exist_ok=True)

print(f"Platform  : {PLATFORM}")
print(f"Work dir  : {WORK_DIR}")
print(f"Ckpt dir  : {CKPT_DIR}")
print(f"Audio dir : {AUDIO_DIR}")

Platform  : kaggle
Work dir  : /kaggle/working
Ckpt dir  : /kaggle/working/checkpoints
Audio dir : /kaggle/working/audio


In [10]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 2 — Install rclone                                ║
# ╚══════════════════════════════════════════════════════════╝

import subprocess

subprocess.run(
    'curl https://rclone.org/install.sh | sudo bash',
    shell=True, capture_output=True
)
ver = subprocess.run('rclone version', shell=True, capture_output=True, text=True)
print(ver.stdout.split('\n')[0])

rclone v1.73.3


In [11]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 3 — Load secrets + configure rclone               ║
# ╚══════════════════════════════════════════════════════════╝
#
# ONE-TIME SETUP — add this secret on Kaggle and Colab:
#   Name  : RCLONE_CONF
#   Value : contents of ~/.config/rclone/rclone.conf on your PC
#           (run:  cat ~/.config/rclone/rclone.conf)
#
# Kaggle : Notebook → Add-ons → Secrets → Add new secret
# Colab  : Left sidebar → key icon → Secrets → Add new secret

import pathlib, re

if ON_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    RCLONE_CONF = UserSecretsClient().get_secret("RCLONE_CONF")
elif ON_COLAB:
    from google.colab import userdata
    RCLONE_CONF = userdata.get('RCLONE_CONF')

# Kaggle Secrets strips newlines — reconstruct proper INI format
raw = RCLONE_CONF.strip()
raw = re.sub(r'\s*(\[[^\]]+\])\s*', r'\n\1\n', raw)
raw = re.sub(r'\s+(type|scope|token|team_drive|client_id|client_secret|'
             r'root_folder_id|service_account_file|drive_id)\s*=\s*',
             r'\n\1 = ', raw)
raw = raw.strip() + '\n'

rclone_cfg = pathlib.Path.home() / '.config/rclone/rclone.conf'
rclone_cfg.parent.mkdir(parents=True, exist_ok=True)
rclone_cfg.write_text(raw)

# Verify
result = subprocess.run('rclone listremotes', shell=True, capture_output=True, text=True)
if 'gdrive:' in result.stdout:
    print("✓ Google Drive connected")
else:
    print("✗ Drive not connected — check RCLONE_CONF secret")
    print(result.stderr[:300])

✓ Google Drive connected


In [12]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 4 — Install ML packages                           ║
# ╚══════════════════════════════════════════════════════════╝

subprocess.run([
    'pip', 'install', '-q',
    'transformers',
    'datasets',
    'torchaudio',
    'speechbrain',
    'peft',
    'librosa',
    'jiwer',
    'evaluate',
], check=True)

print("Packages installed.")

Packages installed.


In [14]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 5 — Pull checkpoints from Drive                   ║
# ╚══════════════════════════════════════════════════════════╝
#
# Run this at the START of every session.
# Downloads all checkpoints saved from previous sessions.

print("Pulling checkpoints from Google Drive...")
result = subprocess.run(
    f'rclone sync gdrive:cse465/checkpoints {CKPT_DIR}',
    shell=True, capture_output=True, text=True
)
if result.returncode != 0:
    print("Warning:", result.stderr[:300])

files = sorted(os.listdir(CKPT_DIR))
if files:
    print(f"\n{len(files)} file(s) in {CKPT_DIR}:")
    for f in files:
        mb = os.path.getsize(f'{CKPT_DIR}/{f}') / 1e6
        print(f"  {f:<50}  {mb:.1f} MB")
else:
    print("No checkpoints yet — fresh start.")

Pulling checkpoints from Google Drive...

1 file(s) in /kaggle/working/checkpoints:
  session_log.json                                    0.0 MB


In [15]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 6 — Checkpoint save / load functions              ║
# ╚══════════════════════════════════════════════════════════╝

import torch, glob, json
from datetime import datetime

def save_checkpoint(state: dict, name: str, step: int, keep: int = 3):
    """
    Saves a checkpoint locally then immediately pushes to Drive.

    Usage:
        save_checkpoint({
            'step'            : step,
            'loss'            : loss.item(),
            'model_state'     : model.state_dict(),
            'optimizer_state' : optimizer.state_dict(),
        }, name='pruning', step=step)
    """
    filename   = f"{name}_step{step:06d}.pt"
    local_path = f"{CKPT_DIR}/{filename}"

    torch.save(state, local_path)
    print(f"[save] {filename}  ({os.path.getsize(local_path)/1e6:.1f} MB)")

    r = subprocess.run(
        f'rclone copy {local_path} gdrive:cse465/checkpoints/',
        shell=True, capture_output=True, text=True
    )
    print(f"[save] Drive push: {'OK' if r.returncode == 0 else 'FAILED — ' + r.stderr[:150]}")

    # Keep only the last `keep` checkpoints locally
    old = sorted(glob.glob(f"{CKPT_DIR}/{name}_step*.pt"))
    for f in old[:-keep]:
        os.remove(f)
        print(f"[save] Removed old local: {os.path.basename(f)}")


def load_latest_checkpoint(name: str):
    """
    Loads the most recent checkpoint for a given name.
    Returns the state dict, or None if nothing found.

    Usage:
        state = load_latest_checkpoint('pruning')
        if state:
            model.load_state_dict(state['model_state'])
            optimizer.load_state_dict(state['optimizer_state'])
            start_step = state['step']
        else:
            start_step = 0
    """
    files = sorted(glob.glob(f"{CKPT_DIR}/{name}_step*.pt"))
    if not files:
        print(f"[load] No checkpoint found for '{name}' — starting fresh.")
        return None
    latest = files[-1]
    state  = torch.load(latest, map_location='cpu')
    print(f"[load] {os.path.basename(latest)}  (step {state.get('step', '?')}, loss {state.get('loss', '?')})")
    return state


def save_model_to_drive(model, processor, stage_name: str):
    """
    Saves a full HuggingFace model to Drive.
    Use at the END of a major stage, not during training.

    Stages you'll use:
        'stage1_teacher_baseline'
        'stage2_pruned'
        'stage3_finetuned'
        'stage4_lora'
        'stage5_final'

    Usage:
        save_model_to_drive(model, processor, 'stage2_pruned')
    """
    local_path = f"{CKPT_DIR}/{stage_name}"
    os.makedirs(local_path, exist_ok=True)

    print(f"[model] Saving to {local_path}...")
    model.save_pretrained(local_path)
    processor.save_pretrained(local_path)

    total_mb = sum(
        os.path.getsize(f"{local_path}/{f}")
        for f in os.listdir(local_path)
    ) / 1e6
    print(f"[model] Size: {total_mb:.0f} MB")

    print(f"[model] Uploading to Drive (may take a few minutes)...")
    r = subprocess.run(
        f'rclone sync {local_path} gdrive:cse465/{stage_name}/',
        shell=True, capture_output=True, text=True
    )
    print(f"[model] Drive upload: {'OK' if r.returncode == 0 else 'FAILED — ' + r.stderr[:150]}")


def load_model_from_drive(stage_name: str):
    """
    Downloads a saved model stage from Drive and loads it.
    Skips download if already present locally.

    Usage:
        model, processor = load_model_from_drive('stage2_pruned')
    """
    from transformers import SeamlessM4Tv2ForSpeechToSpeech, SeamlessM4TProcessor

    local_path = f"{CKPT_DIR}/{stage_name}"

    if os.path.exists(local_path) and os.listdir(local_path):
        print(f"[model] Found locally: {local_path}")
    else:
        print(f"[model] Downloading {stage_name} from Drive...")
        os.makedirs(local_path, exist_ok=True)
        subprocess.run(
            f'rclone sync gdrive:cse465/{stage_name}/ {local_path}',
            shell=True
        )

    print(f"[model] Loading...")
    model     = SeamlessM4Tv2ForSpeechToSpeech.from_pretrained(
                    local_path,
                    torch_dtype=torch.float16,
                    device_map='auto'
                )
    processor = SeamlessM4TProcessor.from_pretrained(local_path)
    print(f"[model] Loaded.")
    return model, processor


print("Functions ready:")
print("  save_checkpoint(state, name, step)")
print("  load_latest_checkpoint(name)")
print("  save_model_to_drive(model, processor, stage_name)")
print("  load_model_from_drive(stage_name)")

Functions ready:
  save_checkpoint(state, name, step)
  load_latest_checkpoint(name)
  save_model_to_drive(model, processor, stage_name)
  load_model_from_drive(stage_name)


In [16]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 7 — Audio play / save helpers                     ║
# ╚══════════════════════════════════════════════════════════╝

import torchaudio
from IPython.display import Audio, display

def play(audio, sr, label=""):
    """
    Play audio inline in the notebook.

    audio : torch.Tensor [1, T] or [T], or numpy array
    sr    : sample rate (int)

    Usage:
        play(waveform, 16000, "Original English")
        play(output_array, model.config.sampling_rate, "Bengali output")
    """
    if hasattr(audio, 'numpy'):
        audio = audio.squeeze().numpy()
    print(f"▶ {label}  ({len(audio)/sr:.1f}s  |  sr={sr})")
    display(Audio(audio, rate=sr))


def save_audio(audio, sr, filename: str, label=""):
    """
    Save audio to the local audio folder AND push to Drive.

    Usage:
        save_audio(waveform, 16000, "original_english.wav", "Original")
        save_audio(output_array, model.config.sampling_rate, "bengali_out.wav", "Bengali")
    """
    path = f"{AUDIO_DIR}/{filename}"

    if hasattr(audio, 'numpy'):
        tensor = audio.squeeze().unsqueeze(0)
        if tensor.dtype != torch.float32:
            tensor = tensor.float()
    else:
        import numpy as np
        tensor = torch.tensor(audio).unsqueeze(0).float()

    torchaudio.save(path, tensor, sr)
    mb = os.path.getsize(path) / 1e6
    print(f"[audio] Saved: {filename}  ({mb:.1f} MB)")

    r = subprocess.run(
        f'rclone copy {path} gdrive:cse465/audio/',
        shell=True, capture_output=True, text=True
    )
    print(f"[audio] Drive push: {'OK' if r.returncode == 0 else 'FAILED'}")


print("Audio functions ready:")
print("  play(audio, sr, label)")
print("  save_audio(audio, sr, filename, label)")

Audio functions ready:
  play(audio, sr, label)
  save_audio(audio, sr, filename, label)


In [17]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 8 — Session status (run anytime to check state)   ║
# ╚══════════════════════════════════════════════════════════╝

def session_status():
    print("=" * 55)
    print(f"  Platform  : {PLATFORM}")
    print(f"  Time      : {datetime.now().strftime('%Y-%m-%d %H:%M')}")

    # Local checkpoints
    local = sorted(glob.glob(f"{CKPT_DIR}/**", recursive=True))
    local = [f for f in local if os.path.isfile(f)]
    print(f"\n  Local files ({len(local)}):")
    for f in local:
        print(f"    {os.path.relpath(f, CKPT_DIR):<45}  {os.path.getsize(f)/1e6:.1f} MB")

    # Drive contents
    drive = subprocess.run(
        'rclone lsf gdrive:cse465/ --recursive',
        shell=True, capture_output=True, text=True
    ).stdout.strip()
    drive_files = [l for l in drive.split('\n') if l.strip()]
    print(f"\n  Drive files ({len(drive_files)}):")
    for f in drive_files:
        print(f"    {f}")

    print("=" * 55)

session_status()

  Platform  : kaggle
  Time      : 2026-04-04 07:19

  Local files (1):
    session_log.json                               0.0 MB

  Drive files (4):
    audio_samples/
    code/
    checkpoints/
    checkpoints/session_log.json


In [18]:
# ╔══════════════════════════════════════════════════════════╗
# ║  LAST CELL — Run before closing every session           ║
# ╚══════════════════════════════════════════════════════════╝

# Push all checkpoints and audio to Drive
print("Final sync to Drive...")
subprocess.run(
    f'rclone sync {CKPT_DIR} gdrive:cse465/checkpoints/',
    shell=True
)
subprocess.run(
    f'rclone sync {AUDIO_DIR} gdrive:cse465/audio/',
    shell=True
)

# Save a session log
log = {
    'platform'  : PLATFORM,
    'time'      : datetime.now().isoformat(),
    'completed' : 'EDIT THIS',   # ← describe what you finished
    'next'      : 'EDIT THIS',   # ← describe what to do next session
    'ckpts'     : [os.path.basename(f)
                   for f in glob.glob(f'{CKPT_DIR}/*.pt')],
}
log_path = f"{CKPT_DIR}/session_log.json"
with open(log_path, 'w') as f:
    json.dump(log, f, indent=2)
subprocess.run(
    f'rclone copy {log_path} gdrive:cse465/checkpoints/',
    shell=True
)

print(json.dumps(log, indent=2))
session_status()

Final sync to Drive...
{
  "platform": "kaggle",
  "time": "2026-04-04T07:19:51.357863",
  "completed": "EDIT THIS",
  "next": "EDIT THIS",
  "ckpts": []
}
  Platform  : kaggle
  Time      : 2026-04-04 07:19

  Local files (1):
    session_log.json                               0.0 MB

  Drive files (4):
    audio_samples/
    code/
    checkpoints/
    checkpoints/session_log.json
